# L09 · Optional Extensions

> *Self-study material — three independent dives that round out your embeddings toolkit. Skip any of them; do them in any order.*

This notebook covers three techniques worth knowing about:

1. **Embeddings for classification** — use sentence embeddings as features for a classifier (instead of a search index)
2. **Multilingual embeddings** — the same model handles English queries against French descriptions
3. **Tokenisation up close** — what subword tokens actually look like for your text

Each takes ~5 minutes.

## Common setup

In [ ]:
# --- Colab setup: install sentence-transformers if missing (no-op in the local dsai-m3 env) ---
try:
    import sentence_transformers  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentence-transformers'], check=True)

import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

torch.set_num_threads(1)
np.random.seed(42)

# Load the product catalogue — local file if present (VS Code), else fetch from GitHub (works in Colab).
import os
_LOCAL = 'data/northstar_catalogue.csv'
_URL = 'https://raw.githubusercontent.com/su-ntu-ctp/6m-data-3.9-Natural-Language-Processing/main/notebooks/data/northstar_catalogue.csv'
df = pd.read_csv(_LOCAL if os.path.exists(_LOCAL) else _URL)
model = SentenceTransformer('all-MiniLM-L6-v2')  # the shared embedding model for the experiments below
print(f"Loaded {len(df)} products")

---

## Experiment 1 — Embeddings as features for classification

We've used embeddings for *retrieval* (find similar things). They're also great as *features* for any downstream classifier. We'll predict product category from description using just a logistic regression on top of the embeddings.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Embed name+description — the 384-number vectors become the features (instead of a search index)
documents = (df['name'] + ' — ' + df['description']).tolist()
X = model.encode(documents, show_progress_bar=False)
y = df['category']  # what we want to PREDICT

print(f"Feature matrix: {X.shape}  (each row is a 384-dim sentence embedding)")
print(f"Number of classes: {y.nunique()}")

# A plain logistic regression on top of the embeddings is enough to classify
clf = LogisticRegression(max_iter=1000)
# cross_val_score measures accuracy fairly by training and testing on different splits.
# cv=3 keeps every fold populated (the smallest category has only ~4 products; cv=5 would warn).
scores = cross_val_score(clf, X, y, cv=3)
print(f"\n3-fold cross-validated accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
print('(Baseline: random guess with 10 classes = 10%)')

**Honest result: ~47% accuracy.** That sounds low, but check the conditions: 10 classes, only ~7 training examples per class in each fold. Random guessing would be 10%. So the model is doing ~5× better than chance using just 7 examples per class — that's the value of pretrained embeddings as features.

With a more realistic 50-200 examples per class, accuracy on this kind of task usually climbs to 85-95%. The bottleneck isn't the embeddings, it's the training-set size.

**Production angle:** for any text-classification task where you have **100-10,000 labelled examples per class**, embed + small classifier is the modern default. You only need to fine-tune the full transformer when you have 100K+ examples per class or a very specialised vocabulary.

In [ ]:
# Fit on all the data, then predict categories for brand-new descriptions it never saw
clf.fit(X, y)
test_sentences = [
    "A lightweight cotton garment perfect for hot days",
    "Heavy-duty rubber-soled outdoor footwear",
    "Plush textile for warming the living room",
    "Compression-fit garment for high-intensity training",
]
test_emb = model.encode(test_sentences, show_progress_bar=False)  # embed novel text the same way
preds = clf.predict(test_emb)
probs = clf.predict_proba(test_emb).max(axis=1)
# Even with only ~7 examples per class it does ~5x better than random — the value of pretrained embeddings as ready-made features
print('Predictions on novel descriptions:')
for s, p, pr in zip(test_sentences, preds, probs):
    print(f"  ({pr:.2f}) {p:12s}  <- {s[:60]}")

---

## Experiment 2 — Multilingual embeddings

Some sentence-transformer models are trained to put the SAME meaning in the SAME place regardless of language. So an English query can retrieve French / Spanish / German documents directly.

Let's load `paraphrase-multilingual-MiniLM-L12-v2` and test.

In [ ]:
# A model trained to place the SAME meaning in the SAME spot regardless of language.
# First run downloads this larger multilingual model (~470 MB) from Hugging Face — give it a minute.
multi = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print('Multilingual model loaded.')

# A small multilingual catalogue snippet — same products described in English, French and Spanish
multi_corpus = [
    ('English', 'A lightweight floral summer dress'),
    ('French',  'Une robe d\'été légère et fleurie'),
    ('Spanish', 'Un vestido de verano ligero con flores'),
    ('English', 'Heavy waterproof hiking boots for trekking'),
    ('French',  'Bottes de randonnée imperméables et robustes'),
    ('Spanish', 'Botas de senderismo impermeables y resistentes'),
    ('English', 'A creamy chocolate dessert with espresso flavour'),
    ('French',  'Un dessert au chocolat crémeux avec un goût d\'expresso'),
]

texts = [t[1] for t in multi_corpus]
embs = multi.encode(texts, show_progress_bar=False)  # one embedding per item, languages mixed together
print(f"Embeddings shape: {embs.shape}")

In [ ]:
# Query in English, retrieve regardless of language —
# because translations land near each other in vector space, one query pulls matching French/Spanish items too
queries = [
    'flowery cotton dress for warm weather',
    'sturdy boots for mountain walks',
    'rich chocolate sweet dish',
]
from sklearn.metrics.pairwise import cosine_similarity

for q in queries:
    q_emb = multi.encode([q])
    sims = cosine_similarity(q_emb, embs)[0]
    order = np.argsort(-sims)[:3]  # top 3 closest items across all languages
    print(f"\nQuery (en): {q!r}")
    for i in order:
        lang, text = multi_corpus[i]
        print(f"  [{lang}] {text}   (score={sims[i]:.3f})")

The model groups translations of the same idea together in vector space, regardless of language. An English query pulls French and Spanish products if they match in meaning. This is invaluable for global e-commerce — one search index, many languages.

**Caveat:** multilingual models are larger (118M params here vs 22M for `all-MiniLM-L6-v2`) and slower. Only use them if you actually need cross-lingual retrieval.

---

## Experiment 3 — Tokenisation up close

What does a sentence-transformer actually see when you call `.encode("blue summer dress")`? Tokens. Specifically, **subword tokens** from a 30K-token vocabulary.

In [ ]:
tokenizer = model.tokenizer

samples = [
    'blue summer dress',
    'lightweight floral frock',
    'P0042',                                # an SKU-like string
    'antidisestablishmentarianism',         # a rare word
    'spaghettification',                    # a rarer compound
    'biryani 🍛',                            # emoji + uncommon word
]

for text in samples:
    # tokenize() reveals what the model actually sees: text split into subword pieces
    # (like spelling unfamiliar words with Scrabble tiles — common words stay whole,
    # rare words and SKUs shatter into fragments, which is why production hybridises with TF-IDF for exact strings)
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text, add_special_tokens=False)  # the integer id behind each token
    print(f"\nText  : {text!r}")
    print(f"Tokens: {tokens}")
    print(f"IDs   : {ids}")

Observations:

- Common words → one token each (`blue`, `summer`, `dress`)
- Rarer words → split into subword pieces (`anti`+`##dis`+`##est`+`##ab`+`##lish`+`##ment`+`##arian`+`##ism`)
- SKU-like strings → fragmented heavily, often confusingly
- Unknown words → still encodable via subwords; the model never says "I don't know that word"

**Why this matters for search:** if your product names contain SKUs, model numbers, or industry jargon, embeddings won't capture them well. That's another reason production systems hybridise with TF-IDF — TF-IDF preserves exact strings perfectly.

---

## Recap

| Technique | Best for |
|-----------|----------|
| Embeddings as classifier features | Any text-classification task with 100-10K labelled examples |
| Multilingual embeddings           | Global e-commerce, customer support across regions |
| Tokenisation awareness            | Debugging surprising search behaviour; SKU / jargon handling |

These three combined cover most of the embedding-related production work you'll encounter. The next lesson (L10) opens the transformer architecture itself — how does the model produce these embeddings in the first place?